In [1]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [2]:
if IN_COLAB:
  # Install dependencies
  ! pip install --upgrade pip
  ! pip install czitools
  ! pip install ipyfilechooser

In [3]:
# import the required libraries
from czitools.metadata_tools import czi_metadata as czimd
from czitools.read_tools import read_tools
from czitools.utils import misc
from ipyfilechooser import FileChooser
from IPython.display import display, HTML
import os
import xarray as xr
import requests
import glob
import ipywidgets as widgets

if not IN_COLAB:
    from czitools.utils.napari_tools import display_xarray_in_napari
    from czitools.utils.napari_tools import display_xarray_list_in_napari


In [ ]:
# try to find the folder with data and download otherwise from GitHub.

# Folder containing the input data
INPUT_FOLDER = 'data/'

# Path to the data on GitHub
GITHUB_DATA_PATH = "https://media.githubusercontent.com/media/sebi06/ZEN_Python_Workshop/main/notebooks/data.zip"

# Download data
if not (os.path.isdir(INPUT_FOLDER)):
    import io
    import zipfile
    # Download training data
    compressed_data = './data.zip'
    if not os.path.isfile(compressed_data):
        print(f"Downloading data from: {GITHUB_DATA_PATH}")
        response = requests.get(GITHUB_DATA_PATH, allow_redirects=True)
        response.raise_for_status()

        # Diagnose if the response is not a real zip (e.g. LFS pointer or HTML error page)
        content = response.content
        print(f"Status: {response.status_code}, Content-Type: {response.headers.get('Content-Type')}, Size: {len(content)} bytes")
        if not content.startswith(b'PK'):
            raise ValueError(
                f"Downloaded content is not a zip file (missing PK header).\n"
                f"First 200 bytes: {content[:200]}"
            )

        compressed_data = io.BytesIO(content)

    with zipfile.ZipFile(compressed_data, 'r') as zip_accessor:
        zip_accessor.extractall('./')
        print(f"Extracted: {zip_accessor.namelist()}")

In [5]:
if not IN_COLAB:
    # choose local file
    fc = FileChooser()
    fc.default_path = INPUT_FOLDER
    fc.filter_pattern = '*.czi'
    display(fc)

elif IN_COLAB:
    # list files inside the folder on gdrive
    czifiles = glob.glob(os.path.join(INPUT_FOLDER, "*.czi"))
    wd = widgets.Select(
        options=czifiles,
        description='CZI Files:',
        layout={'width': 'max-content'}
    )
    display(wd)

FileChooser(path='F:\GitHub\ZEN_Python_Workshop\notebooks\data', filename='', title='', show_hidden=False, sel…

In [6]:
if not IN_COLAB:
    filepath = fc.selected
elif IN_COLAB:
    filepath = wd.value

print(f"Selected File: {filepath}")

Selected File: F:\GitHub\ZEN_Python_Workshop\notebooks\data\T=3_Z=5_CH=2_X=240_Y=170.czi


In [7]:
# get the complete metadata at once as one big class
mdata = czimd.CziMetadata(filepath)

# convert metadata dictionary to a pandas dataframe
mdframe = misc.md2dataframe(mdata, reduced_params=True)

# create a ipywdiget to show the dataframe with the metadata
wd = widgets.Output(layout={"scrollY": "auto", "height": "300px"})

with wd:
    display(HTML(mdframe.to_html()))
display(widgets.VBox(children=[wd]))

  0% |                                                  | ETA:  --:--:-- 0 of 30
100% |#################################################| Time:  0:00:00 30 of 30


In [8]:
result, dims, num_stacks, mdata = read_tools.read_stacks(
    filepath,
    use_dask=True,
    use_xarray=True,
    stack_scenes=True,
)

if isinstance(result, list):
    # List of per-scene arrays
    for idx, arr in enumerate(result):
        if isinstance(arr, xr.DataArray):
            print(f"Stack {idx}: dims={arr.dims}, shape={arr.shape}, dtype={arr.dtype}")
        else:
            print(f"Stack {idx}: shape={arr.shape}, dims={dims}, dtype={arr.dtype}")
else:
    # Single stacked array
    if isinstance(result, xr.DataArray):
        print(f"Stacked: dims={result.dims}, shape={result.shape}, dtype={result.dtype}")
    else:
        print(f"Stacked: shape={result.shape}, dims={dims}, dtype={result.dtype}")

    # With use_dask=True, result is backed by dask - no data loaded yet
    print(f"\nArray shape (no data loaded): {result.shape}")


  0% |                                                  | ETA:  --:--:-- 0 of 30
100% |#################################################| Time:  0:00:00 30 of 30


2026-05-17 18:49:43,102 - czitools - INFO - read_stacks: num_stacks=1 (selected from total=1), total_bounding_box={'T': (0, 3), 'Z': (0, 5), 'C': (0, 2), 'B': (0, 1), 'X': (0, 240), 'Y': (0, 170)}
2026-05-17 18:49:43,103 - czitools - INFO - read_stacks: canonical_dims=['T', 'C', 'Z'], dim_sizes={'T': 3, 'C': 2, 'Z': 5, 'S': 1}
2026-05-17 18:49:43,103 - czitools - INFO - read_stacks: Using lazy dask arrays - data will be read on demand
2026-05-17 18:49:43,198 - czitools - INFO - read_stacks: Stacking 1 stacks (all shapes equal: (3, 2, 5, 170, 240))
Stacked: dims=('S', 'T', 'C', 'Z', 'Y', 'X'), shape=(1, 3, 2, 5, 170, 240), dtype=uint16

Array shape (no data loaded): (1, 3, 2, 5, 170, 240)


In [9]:
if not IN_COLAB:

    # If result is a list of stacks, choose display mode:
    # - True: show all stacks in one viewer
    # - False: show only one stack selected by ``list_stack_index``
    show_all_list_stacks = True
    # Stack index used when ``result`` is a list and ``show_all_list_stacks`` is False.
    # Ignored when result is not a list.
    list_stack_index = 0

    if isinstance(result, list):
        if len(result) == 0:
            raise RuntimeError("No stack data available for Napari display.")

        if show_all_list_stacks:
            display_xarray_list_in_napari(result, mdata)
        else:
            selected_index = max(0, min(list_stack_index, len(result) - 1))
            stack = result[selected_index]
            subset_planes = stack.attrs.get("subset_planes", {}) if isinstance(stack, xr.DataArray) else {}
            print(f"Showing stack index: {selected_index}")
            display_xarray_in_napari(stack, mdata, subset_planes)
    else:
        subset_planes = result.attrs.get("subset_planes", {}) if isinstance(result, xr.DataArray) else {}
        display_xarray_in_napari(result, mdata, subset_planes)

In [ ]:
napari.utils.nbscreenshot(viewer)